# DIF Analysis: AI Impact Hypothesis Validation

Uses **Differential Item Functioning (DIF)** to test whether skill demands shifted
after the ChatGPT release (2022Q4), controlling for JD complexity.

**5 Hypotheses** (from `ai_impact_dimensions.json`):
1. **Replaced by AI** → declining demand (negative DIF)
2. **Augmented by AI** → increasing demand (positive DIF)
3. **Building/Managing AI** → strongest growth (largest positive DIF)
4. **Resistant to AI** → stable demand (negligible DIF)
5. **Transformed by AI** → mixed DIF (workflow restructuring)

**Methods**: Mantel-Haenszel (MH) with ETS delta classification + Logistic Regression DIF

In [ ]:
# =============================================================================
# Cell 1: Configuration & Imports
# =============================================================================
import subprocess, sys
for pkg in ['scipy', 'statsmodels']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
import json
import ast
import gc
from typing import List, Dict, Tuple
import pyarrow.parquet as pq

from scipy import stats as scipy_stats
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_FOLDER = '../data/extracted_skills_production'
SKILLS_COLUMN = 'skills'
DATE_COLUMN = 'post_date'

# DIF settings
SPLIT_QUARTER = '2022Q4'        # ChatGPT release boundary
MIN_QUARTER = '2022Q1'          # Only include data from this quarter onwards
MIN_SKILL_FREQ = 100            # Minimum total mentions to include a skill in DIF
N_STRATA = 5                    # Quintile bins for matching variable
ALPHA = 0.05                    # Significance level

# Load AI Impact lookup
AI_IMPACT_LOOKUP_PATH = Path('../skillner/data/ksao_ai_impact_lookup.json')
AI_DIMENSIONS_PATH = Path('../skillner/data/ai_impact_dimensions.json')

with open(AI_IMPACT_LOOKUP_PATH) as f:
    AI_IMPACT_LOOKUP = json.load(f)
with open(AI_DIMENSIONS_PATH) as f:
    AI_DIMENSIONS = json.load(f)

# Find parquet files
input_path = Path(INPUT_FOLDER)
parquet_files = sorted(input_path.glob('*.parquet'))

print(f"\u2713 {len(parquet_files)} parquet files found")
print(f"\u2713 {len(AI_IMPACT_LOOKUP):,} KSAOs in AI Impact lookup")
print(f"\u2713 Split point: {SPLIT_QUARTER} (ChatGPT release)")
print(f"\u2713 Data range: {MIN_QUARTER}+")

In [ ]:
# =============================================================================
# Cell 2: Helper Functions
# =============================================================================

def parse_skills_column(value) -> List[str]:
    """Parse skills column (string, list, array, None)."""
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        return [str(s) for s in value if s is not None]
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        if value in ('[]', '', 'nan', 'None'):
            return []
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            pass
    return []


def date_to_quarter(dt) -> str:
    """Convert datetime to quarter string like '2023Q2'."""
    if pd.isna(dt):
        return None
    return f"{dt.year}Q{(dt.month - 1) // 3 + 1}"


def assign_period(quarter_str: str) -> str:
    """Assign 'pre' or 'post' based on SPLIT_QUARTER."""
    if quarter_str is None:
        return None
    return 'post' if quarter_str >= SPLIT_QUARTER else 'pre'


def get_ai_dimension(skill: str) -> str:
    """Look up AI Impact Dimension for a skill."""
    skill_lower = skill.lower().strip()
    if skill_lower in AI_IMPACT_LOOKUP:
        return AI_IMPACT_LOOKUP[skill_lower]['ai_impact_dimension']
    return None


print("\u2713 Helper functions defined")
print(f"  assign_period('2022Q3') = '{assign_period('2022Q3')}'")
print(f"  assign_period('2023Q1') = '{assign_period('2023Q1')}'")

In [ ]:
# =============================================================================
# Cell 3: Streaming Pass 1 — Skill Frequencies & Total Skills Distribution
# =============================================================================
print("Pass 1: Collecting skill frequencies and JD complexity distribution...")

skill_freq = Counter()           # skill -> total count
total_skills_list = []           # list of num_skills per JD (for quintile computation)
quarter_jd_counts = Counter()    # quarter -> num JDs
period_jd_counts = Counter()     # 'pre'/'post' -> num JDs

for i, filepath in enumerate(parquet_files):
    try:
        df = pd.read_parquet(filepath, columns=[SKILLS_COLUMN, DATE_COLUMN])
    except Exception as e:
        print(f"  Skipping {filepath.name}: {e}")
        continue

    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors='coerce')

    for _, row in df.iterrows():
        q = date_to_quarter(row[DATE_COLUMN])
        if q is None or q < MIN_QUARTER:
            continue

        skills = parse_skills_column(row[SKILLS_COLUMN])
        n_skills = len(skills)

        quarter_jd_counts[q] += 1
        period_jd_counts[assign_period(q)] += 1
        total_skills_list.append(n_skills)
        skill_freq.update(skills)

    del df
    gc.collect()
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(parquet_files)}]")

# Compute quintile boundaries for matching variable
total_skills_arr = np.array(total_skills_list)
quintile_boundaries = np.percentile(total_skills_arr, [20, 40, 60, 80])

print(f"\n\u2713 Pass 1 complete")
print(f"  Total JDs (from {MIN_QUARTER}): {sum(quarter_jd_counts.values()):,}")
print(f"  Pre-ChatGPT JDs: {period_jd_counts['pre']:,}")
print(f"  Post-ChatGPT JDs: {period_jd_counts['post']:,}")
print(f"  Unique skills: {len(skill_freq):,}")
print(f"  Quintile boundaries (total_skills): {quintile_boundaries}")
print(f"  Quarters: {sorted(quarter_jd_counts.keys())}")

In [ ]:
# =============================================================================
# Cell 4: Select Items for DIF Analysis
# =============================================================================

# Filter skills with enough frequency AND known AI dimension
dif_skills = {}
for skill, freq in skill_freq.items():
    if freq >= MIN_SKILL_FREQ:
        dim = get_ai_dimension(skill)
        if dim is not None:
            dif_skills[skill] = {'freq': freq, 'dimension': dim}

# Summary by dimension
dim_skill_counts = Counter(v['dimension'] for v in dif_skills.values())

print(f"Skills selected for DIF analysis: {len(dif_skills):,}")
print(f"(min frequency = {MIN_SKILL_FREQ}, must have known AI dimension)\n")
print(f"{'AI Impact Dimension':<30} {'# Skills':>10}")
print("-" * 42)
for dim in ['Replaced by AI', 'Augmented by AI', 'Building/Managing AI',
            'Resistant to AI', 'Transformed by AI']:
    # Handle dimension name variations
    count = dim_skill_counts.get(dim, 0)
    # Also check 'Building & Managing AI' variant
    if count == 0:
        count = dim_skill_counts.get(dim.replace('/', ' & ').replace('/Managing', ' & Managing'), 0)
    print(f"{dim:<30} {count:>10,}")

# Build set for fast lookup in Pass 2
dif_skill_set = set(dif_skills.keys())

In [ ]:
# =============================================================================
# Cell 5: Streaming Pass 2 — Build Contingency Tables
# =============================================================================
# For each skill, build a 2x2 table per stratum:
#   rows: skill present/absent
#   cols: pre/post ChatGPT
#   stratified by: total_skills quintile bin

print("Pass 2: Building contingency tables for DIF analysis...")

def get_stratum(n_skills: int) -> int:
    """Assign JD to quintile stratum based on total skill count."""
    for k, boundary in enumerate(quintile_boundaries):
        if n_skills <= boundary:
            return k
    return len(quintile_boundaries)

# contingency[skill][stratum] = {'pre_yes': 0, 'pre_no': 0, 'post_yes': 0, 'post_no': 0}
contingency = {skill: {s: {'pre_yes': 0, 'pre_no': 0, 'post_yes': 0, 'post_no': 0}
                        for s in range(N_STRATA)}
               for skill in dif_skill_set}

# Also collect quarterly mention rates for trend visualization
# quarterly_skill_counts[skill][quarter] = count of JDs mentioning that skill
quarterly_skill_counts = {skill: Counter() for skill in dif_skill_set}

for i, filepath in enumerate(parquet_files):
    try:
        df = pd.read_parquet(filepath, columns=[SKILLS_COLUMN, DATE_COLUMN])
    except Exception:
        continue

    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors='coerce')

    for _, row in df.iterrows():
        q = date_to_quarter(row[DATE_COLUMN])
        if q is None or q < MIN_QUARTER:
            continue

        period = assign_period(q)
        skills = parse_skills_column(row[SKILLS_COLUMN])
        n_skills = len(skills)
        stratum = get_stratum(n_skills)
        skills_set = set(skills)

        for skill in dif_skill_set:
            present = skill in skills_set
            if present:
                contingency[skill][stratum][f'{period}_yes'] += 1
                quarterly_skill_counts[skill][q] += 1
            else:
                contingency[skill][stratum][f'{period}_no'] += 1

    del df
    gc.collect()
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(parquet_files)}]")

print(f"\n\u2713 Pass 2 complete. Contingency tables built for {len(contingency):,} skills.")

In [ ]:
# =============================================================================
# Cell 6: Mantel-Haenszel DIF Analysis
# =============================================================================
print("Running Mantel-Haenszel DIF analysis...\n")

dif_results = []

for skill in dif_skill_set:
    tables = contingency[skill]

    # Build stratum arrays for MH computation
    numerator_or = 0.0   # sum of a*d/n per stratum
    denominator_or = 0.0 # sum of b*c/n per stratum
    numerator_chi = 0.0  # sum of (a - E(a)) per stratum
    variance_sum = 0.0   # sum of Var(a) per stratum

    valid_strata = 0

    for s in range(N_STRATA):
        t = tables[s]
        a = t['pre_yes']   # pre + present
        b = t['post_yes']  # post + present
        c = t['pre_no']    # pre + absent
        d = t['post_no']   # post + absent
        n = a + b + c + d

        if n == 0:
            continue

        valid_strata += 1

        # MH odds ratio components
        numerator_or += (a * d) / n
        denominator_or += (b * c) / n

        # MH chi-square components
        row1 = a + b  # skill present
        col1 = a + c  # pre period
        e_a = (row1 * col1) / n if n > 0 else 0
        numerator_chi += a - e_a

        # Variance of a under null
        row2 = c + d  # skill absent
        col2 = b + d  # post period
        if n > 1:
            var_a = (row1 * row2 * col1 * col2) / (n * n * (n - 1))
        else:
            var_a = 0
        variance_sum += var_a

    if valid_strata == 0 or denominator_or == 0 or variance_sum == 0:
        continue

    # MH odds ratio
    mh_or = numerator_or / denominator_or

    # MH chi-square (with continuity correction)
    chi2_mh = (abs(numerator_chi) - 0.5) ** 2 / variance_sum
    p_value = 1 - scipy_stats.chi2.cdf(chi2_mh, df=1)

    # ETS delta scale: delta_MH = -2.35 * ln(MH_OR)
    # Positive delta = skill more common in REFERENCE (pre) group
    # We flip sign so positive = increasing post-ChatGPT
    delta_mh = 2.35 * np.log(mh_or)  # positive = more common post-ChatGPT

    # DIF classification (ETS)
    abs_delta = abs(delta_mh)
    if abs_delta < 1.0:
        dif_class = 'A'  # negligible
    elif abs_delta < 1.5:
        dif_class = 'B'  # moderate
    else:
        dif_class = 'C'  # large

    direction = 'increasing' if delta_mh > 0 else 'declining' if delta_mh < 0 else 'stable'
    dimension = dif_skills[skill]['dimension']

    dif_results.append({
        'skill': skill,
        'dimension': dimension,
        'freq': dif_skills[skill]['freq'],
        'mh_or': mh_or,
        'delta_mh': delta_mh,
        'chi2_mh': chi2_mh,
        'p_value': p_value,
        'dif_class': dif_class,
        'direction': direction,
        'significant': p_value < ALPHA,
    })

dif_df = pd.DataFrame(dif_results)

# Apply Benjamini-Hochberg FDR correction
from statsmodels.stats.multitest import multipletests
if len(dif_df) > 0:
    reject, p_adj, _, _ = multipletests(dif_df['p_value'], alpha=ALPHA, method='fdr_bh')
    dif_df['p_adjusted'] = p_adj
    dif_df['significant_fdr'] = reject

print(f"\u2713 MH DIF computed for {len(dif_df):,} skills")
print(f"\nDIF Classification Summary:")
print(dif_df['dif_class'].value_counts().to_string())
print(f"\nSignificant (FDR-adjusted): {dif_df['significant_fdr'].sum():,} / {len(dif_df):,}")
print(f"\nTop 10 skills with largest |delta_MH|:")
top10 = dif_df.nlargest(10, 'delta_mh', keep='first')[['skill', 'dimension', 'delta_mh', 'dif_class', 'p_adjusted']]
print(top10.to_string(index=False))
print(f"\nBottom 10 skills with most negative delta_MH:")
bot10 = dif_df.nsmallest(10, 'delta_mh', keep='first')[['skill', 'dimension', 'delta_mh', 'dif_class', 'p_adjusted']]
print(bot10.to_string(index=False))

In [ ]:
# =============================================================================
# Cell 7: Logistic Regression DIF (for B/C classified skills)
# =============================================================================
# Nested model comparison (more rigorous than single-model approach):
#   Model 1: logit(P) = b0 + b1*stratum                          (no DIF)
#   Model 2: logit(P) = b0 + b1*stratum + b2*period              (uniform DIF)
#   Model 3: logit(P) = b0 + b1*stratum + b2*period + b3*period*stratum  (non-uniform DIF)
#
# Uniform DIF test: LR chi-square Model2 vs Model1 (1 df)
# Non-uniform DIF test: LR chi-square Model3 vs Model2 (1 df)

bc_skills = dif_df[dif_df['dif_class'].isin(['B', 'C'])]['skill'].tolist()
print(f"Running Logistic Regression DIF for {len(bc_skills)} B/C-classified skills...\n")

# Stricter alpha for LR (large N makes everything significant)
LR_ALPHA = 0.01
MIN_EFFECT_SIZE = 0.10  # minimum |beta| for practical significance

lr_results = []

for skill in bc_skills:
    tables = contingency[skill]

    # Reconstruct aggregated data from contingency tables
    rows = []
    for s in range(N_STRATA):
        t = tables[s]
        for period_code, present, count in [
            (0, 1, t['pre_yes']),
            (0, 0, t['pre_no']),
            (1, 1, t['post_yes']),
            (1, 0, t['post_no']),
        ]:
            if count > 0:
                rows.append({
                    'present': present,
                    'period': period_code,  # 0=pre, 1=post
                    'stratum': s,
                    'weight': count
                })

    if not rows:
        continue

    lr_data = pd.DataFrame(rows)

    try:
        y = lr_data['present']
        weights = lr_data['weight']

        # Model 1: no DIF (stratum only)
        X1 = sm.add_constant(lr_data[['stratum']])
        m1 = sm.GLM(y, X1, family=sm.families.Binomial(), freq_weights=weights).fit(disp=0)

        # Model 2: uniform DIF (+ period)
        X2 = sm.add_constant(lr_data[['stratum', 'period']])
        m2 = sm.GLM(y, X2, family=sm.families.Binomial(), freq_weights=weights).fit(disp=0)

        # Model 3: non-uniform DIF (+ period*stratum interaction)
        X3_data = lr_data[['stratum', 'period']].copy()
        X3_data['period_x_stratum'] = X3_data['period'] * X3_data['stratum']
        X3 = sm.add_constant(X3_data)
        m3 = sm.GLM(y, X3, family=sm.families.Binomial(), freq_weights=weights).fit(disp=0)

        # Likelihood ratio tests
        lr_uniform = m1.deviance - m2.deviance      # Model1 vs Model2 (1 df)
        lr_nonuniform = m2.deviance - m3.deviance    # Model2 vs Model3 (1 df)
        lr_total = m1.deviance - m3.deviance         # Model1 vs Model3 (2 df)

        p_uniform = 1 - scipy_stats.chi2.cdf(lr_uniform, 1)
        p_nonuniform = 1 - scipy_stats.chi2.cdf(lr_nonuniform, 1)
        p_total = 1 - scipy_stats.chi2.cdf(lr_total, 2)

        period_coef = m2.params.get('period', np.nan)
        period_or = np.exp(period_coef) if not np.isnan(period_coef) else np.nan
        interaction_coef = m3.params.get('period_x_stratum', np.nan)

        lr_results.append({
            'skill': skill,
            'dimension': dif_skills[skill]['dimension'],
            'period_coef': period_coef,
            'period_or': period_or,
            'interaction_coef': interaction_coef,
            'lr_chi2_uniform': lr_uniform,
            'p_uniform': p_uniform,
            'lr_chi2_nonuniform': lr_nonuniform,
            'p_nonuniform': p_nonuniform,
            'lr_chi2_total': lr_total,
            'p_total': p_total,
            'uniform_dif': p_uniform < LR_ALPHA and abs(period_coef) >= MIN_EFFECT_SIZE,
            'nonuniform_dif': p_nonuniform < LR_ALPHA and abs(interaction_coef) >= MIN_EFFECT_SIZE,
        })
    except Exception:
        continue

lr_df_results = pd.DataFrame(lr_results)

if len(lr_df_results) > 0:
    print(f"\u2713 Logistic Regression DIF computed for {len(lr_df_results)} skills")
    print(f"  (alpha={LR_ALPHA}, min |beta|={MIN_EFFECT_SIZE})")
    print(f"\nUniform DIF (sig + practical): {lr_df_results['uniform_dif'].sum()} skills")
    print(f"Non-uniform DIF (sig + practical): {lr_df_results['nonuniform_dif'].sum()} skills")
    print(f"\nTop 10 by period coefficient (increasing post-ChatGPT):")
    top_lr = lr_df_results.nlargest(10, 'period_coef')[
        ['skill', 'dimension', 'period_coef', 'period_or', 'p_uniform', 'uniform_dif']]
    print(top_lr.to_string(index=False))
    print(f"\nSkills with non-uniform DIF (effect varies by JD complexity):")
    nu = lr_df_results[lr_df_results['nonuniform_dif']][
        ['skill', 'dimension', 'interaction_coef', 'p_nonuniform']]
    if len(nu) > 0:
        print(nu.to_string(index=False))
    else:
        print("  None detected")
else:
    print("No B/C skills found or all regressions failed.")

In [ ]:
# =============================================================================
# Cell 8: Aggregate Results by AI Impact Dimension
# =============================================================================
print("="*70)
print("DIF RESULTS AGGREGATED BY AI IMPACT DIMENSION")
print("="*70)

# Get all unique dimension names from results
all_dimensions = dif_df['dimension'].unique()

dimension_summary = []

for dim in sorted(all_dimensions):
    dim_data = dif_df[dif_df['dimension'] == dim]
    n_skills = len(dim_data)
    if n_skills == 0:
        continue

    mean_delta = dim_data['delta_mh'].mean()
    median_delta = dim_data['delta_mh'].median()
    std_delta = dim_data['delta_mh'].std()

    n_significant = dim_data['significant_fdr'].sum()
    pct_significant = n_significant / n_skills * 100

    n_increasing = (dim_data['delta_mh'] > 0).sum()
    n_declining = (dim_data['delta_mh'] < 0).sum()
    pct_increasing = n_increasing / n_skills * 100

    n_class_b = (dim_data['dif_class'] == 'B').sum()
    n_class_c = (dim_data['dif_class'] == 'C').sum()

    # One-sample t-test: is mean delta significantly different from 0?
    if n_skills >= 3:
        t_stat, t_pval = scipy_stats.ttest_1samp(dim_data['delta_mh'], 0)
    else:
        t_stat, t_pval = np.nan, np.nan

    dimension_summary.append({
        'dimension': dim,
        'n_skills': n_skills,
        'mean_delta': mean_delta,
        'median_delta': median_delta,
        'std_delta': std_delta,
        'n_significant_fdr': int(n_significant),
        'pct_significant': pct_significant,
        'pct_increasing': pct_increasing,
        'n_class_B': int(n_class_b),
        'n_class_C': int(n_class_c),
        't_stat': t_stat,
        't_pval': t_pval,
    })

dim_summary_df = pd.DataFrame(dimension_summary)

print(f"\n{'Dimension':<30} {'N':>5} {'Mean \u0394':>8} {'Med \u0394':>8} {'%Sig':>6} {'%Inc':>6} {'B':>4} {'C':>4} {'t-test p':>10}")
print("-" * 95)
for _, row in dim_summary_df.iterrows():
    print(f"{row['dimension']:<30} {row['n_skills']:>5} {row['mean_delta']:>8.2f} {row['median_delta']:>8.2f} "
          f"{row['pct_significant']:>5.1f}% {row['pct_increasing']:>5.1f}% {row['n_class_B']:>4} {row['n_class_C']:>4} "
          f"{row['t_pval']:>10.4f}")

In [ ]:
# =============================================================================
# Cell 9: Hypothesis Validation Summary
# =============================================================================
from scipy.stats import binomtest

print("="*70)
print("HYPOTHESIS VALIDATION SUMMARY")
print("="*70)

hypotheses = {
    'Replaced by AI': {
        'prediction': 'negative delta (declining demand)',
        'expected_dir': 'declining',
        'check': lambda row: row['mean_delta'] < 0 and row['t_pval'] < 0.05,
        'partial': lambda row: row['mean_delta'] < 0,
    },
    'Augmented by AI': {
        'prediction': 'positive delta (increasing demand)',
        'expected_dir': 'increasing',
        'check': lambda row: row['mean_delta'] > 0 and row['t_pval'] < 0.05,
        'partial': lambda row: row['mean_delta'] > 0,
    },
    'Building & Managing AI': {
        'prediction': 'strongest positive delta (highest growth)',
        'expected_dir': 'increasing',
        'check': lambda row: row['mean_delta'] > 0 and row['t_pval'] < 0.05,
        'partial': lambda row: row['mean_delta'] > 0,
    },
    'Resistant to AI': {
        'prediction': 'near-zero delta (stable demand)',
        'expected_dir': 'stable',
        'check': lambda row: row['t_pval'] >= 0.05,  # not significantly different from 0
        'partial': lambda row: abs(row['mean_delta']) < 0.5,
    },
    'Transformed by AI': {
        'prediction': 'mixed direction (workflow restructuring)',
        'expected_dir': 'mixed',
        'check': lambda row: 30 < row['pct_increasing'] < 70,  # mixed directions
        'partial': lambda row: True,
    },
}

for dim_name, hyp in hypotheses.items():
    row = dim_summary_df[dim_summary_df['dimension'].str.contains(dim_name.split(' ')[0], case=False)]
    if len(row) == 0:
        print(f"\n  {dim_name}: NO DATA")
        continue

    row = row.iloc[0]
    supported = hyp['check'](row)
    partial = hyp['partial'](row)

    if supported:
        verdict = '\u2705 SUPPORTED'
    elif partial:
        verdict = '\u26a0\ufe0f  PARTIALLY SUPPORTED'
    else:
        verdict = '\u274c NOT SUPPORTED'

    # Binomial directional consistency test
    dim_data = dif_df[dif_df['dimension'] == row['dimension']]
    sig_skills = dim_data[dim_data['significant_fdr']]
    n_sig = len(sig_skills)
    binom_str = ""
    if n_sig >= 3 and hyp['expected_dir'] in ('increasing', 'declining'):
        if hyp['expected_dir'] == 'declining':
            successes = (sig_skills['delta_mh'] < 0).sum()
        else:
            successes = (sig_skills['delta_mh'] > 0).sum()
        bt = binomtest(int(successes), n_sig, 0.5, alternative='greater')
        consistency = successes / n_sig * 100
        binom_str = f"\n   Directional consistency: {successes}/{n_sig} ({consistency:.0f}%) in expected direction, binomial p={bt.pvalue:.4f}"

    print(f"\nH: {dim_name} \u2014 {verdict}")
    print(f"   Prediction: {hyp['prediction']}")
    print(f"   Observed: Mean \u0394_MH = {row['mean_delta']:.3f}, "
          f"{row['pct_increasing']:.0f}% increasing, "
          f"{row['pct_significant']:.0f}% significant (FDR), "
          f"t-test p = {row['t_pval']:.4f}")
    print(f"   Skills analyzed: {row['n_skills']}, "
          f"B-level DIF: {row['n_class_B']}, C-level DIF: {row['n_class_C']}")
    if binom_str:
        print(binom_str)

# Note about large-N
print("\n" + "-"*70)
print("NOTE: With large datasets, statistical significance is near-guaranteed.")
print("Focus on effect sizes (delta_MH, DIF class B/C) and directional patterns")
print("rather than p-values alone.")

In [ ]:
# =============================================================================
# Cell 10: Visualization — DIF Effect Sizes by Dimension
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

dim_colors = {
    'Replaced by AI': '#e74c3c',
    'Augmented by AI': '#3498db',
    'Building & Managing AI': '#9b59b6',
    'Building/Managing AI': '#9b59b6',
    'Resistant to AI': '#27ae60',
    'Transformed by AI': '#f39c12',
}

# 1. Box plot: delta_MH distribution per dimension
ax = axes[0]
dim_order = sorted(dif_df['dimension'].unique())
colors = [dim_colors.get(d, '#95a5a6') for d in dim_order]
box_data = [dif_df[dif_df['dimension'] == d]['delta_mh'].values for d in dim_order]
bp = ax.boxplot(box_data, labels=[d.replace(' ', '\n') for d in dim_order],
                patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.axhline(1.0, color='red', linestyle=':', alpha=0.3, label='B threshold')
ax.axhline(-1.0, color='red', linestyle=':', alpha=0.3)
ax.set_ylabel('\u0394_MH (ETS Delta Scale)')
ax.set_title('DIF Effect Size Distribution by AI Dimension')
ax.tick_params(axis='x', rotation=0, labelsize=8)

# 2. Volcano plot: delta_MH vs -log10(p_adjusted)
ax = axes[1]
for dim in dim_order:
    mask = dif_df['dimension'] == dim
    subset = dif_df[mask]
    neg_log_p = -np.log10(subset['p_adjusted'].clip(lower=1e-300))
    ax.scatter(subset['delta_mh'], neg_log_p,
              c=dim_colors.get(dim, '#95a5a6'), label=dim,
              alpha=0.5, s=20, edgecolors='none')
ax.axhline(-np.log10(ALPHA), color='red', linestyle='--', alpha=0.5, label=f'p={ALPHA}')
ax.axvline(0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('\u0394_MH (+ = increasing post-ChatGPT)')
ax.set_ylabel('-log10(p adjusted)')
ax.set_title('Volcano Plot: DIF Significance vs Effect Size')
ax.legend(fontsize=7, loc='upper right')

# 3. Summary bar: mean delta per dimension
ax = axes[2]
means = dim_summary_df.set_index('dimension')['mean_delta']
bar_colors = [dim_colors.get(d, '#95a5a6') for d in means.index]
bars = ax.barh(range(len(means)), means.values, color=bar_colors, edgecolor='black', alpha=0.8)
ax.set_yticks(range(len(means)))
ax.set_yticklabels(means.index, fontsize=9)
ax.axvline(0, color='black', linestyle='-', alpha=0.5)
ax.set_xlabel('Mean \u0394_MH')
ax.set_title('Mean DIF Effect by AI Dimension')
ax.grid(axis='x', alpha=0.3)
for i, (val, p) in enumerate(zip(means.values, dim_summary_df.set_index('dimension')['t_pval'])):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.text(val + 0.05 * np.sign(val), i, f'{val:.2f} {sig}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Cell 11: Quarterly Trend for Top DIF-Flagged Skills
# =============================================================================
# Show quarterly mention rate for the top 3 DIF skills per dimension

dims_to_plot = sorted(dif_df['dimension'].unique())
n_dims = len(dims_to_plot)
fig, axes = plt.subplots(n_dims, 1, figsize=(14, 4 * n_dims), sharex=True)
if n_dims == 1:
    axes = [axes]

all_quarters = sorted(quarter_jd_counts.keys())
split_idx = all_quarters.index(SPLIT_QUARTER) if SPLIT_QUARTER in all_quarters else None

for ax, dim in zip(axes, dims_to_plot):
    dim_skills = dif_df[dif_df['dimension'] == dim].nlargest(3, 'delta_mh', keep='first')
    dim_skills_neg = dif_df[dif_df['dimension'] == dim].nsmallest(3, 'delta_mh', keep='first')
    top_skills = pd.concat([dim_skills, dim_skills_neg]).drop_duplicates('skill').head(5)

    for _, row in top_skills.iterrows():
        skill = row['skill']
        rates = []
        for q in all_quarters:
            total = quarter_jd_counts[q]
            mentions = quarterly_skill_counts[skill].get(q, 0)
            rates.append(mentions / total * 100 if total > 0 else 0)

        label = f"{skill[:30]} (\u0394={row['delta_mh']:.2f})"
        ax.plot(range(len(all_quarters)), rates, marker='o', markersize=3,
                linewidth=1.5, label=label)

    if split_idx is not None:
        ax.axvline(split_idx, color='red', linestyle='--', alpha=0.7, label='ChatGPT Release')

    ax.set_ylabel('Mention Rate (%)')
    ax.set_title(f'{dim}', fontsize=11, fontweight='bold',
                 color=dim_colors.get(dim, 'black'))
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)

axes[-1].set_xticks(range(0, len(all_quarters), max(1, len(all_quarters)//15)))
axes[-1].set_xticklabels(
    [all_quarters[i] for i in range(0, len(all_quarters), max(1, len(all_quarters)//15))],
    rotation=45, ha='right', fontsize=9)
axes[-1].set_xlabel('Quarter')

plt.suptitle('Quarterly Skill Mention Rates: Top DIF-Flagged Skills by Dimension',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Cell 12: Export Results
# =============================================================================
output_dir = Path(INPUT_FOLDER)

# Save full DIF results
dif_csv_path = output_dir / 'dif_results.csv'
dif_df.to_csv(dif_csv_path, index=False)
print(f"\u2713 DIF results saved to: {dif_csv_path}")

# Save logistic regression results
if len(lr_df_results) > 0:
    lr_csv_path = output_dir / 'dif_logistic_regression_results.csv'
    lr_df_results.to_csv(lr_csv_path, index=False)
    print(f"\u2713 LR DIF results saved to: {lr_csv_path}")

# Save dimension summary as JSON
summary = {
    'analysis': 'Differential Item Functioning (DIF) - AI Impact Hypothesis Validation',
    'split_point': SPLIT_QUARTER,
    'min_quarter': MIN_QUARTER,
    'min_skill_freq': MIN_SKILL_FREQ,
    'n_strata': N_STRATA,
    'total_skills_analyzed': len(dif_df),
    'pre_jds': int(period_jd_counts['pre']),
    'post_jds': int(period_jd_counts['post']),
    'dimensions': {}
}

for _, row in dim_summary_df.iterrows():
    summary['dimensions'][row['dimension']] = {
        'n_skills': int(row['n_skills']),
        'mean_delta_mh': round(float(row['mean_delta']), 4),
        'median_delta_mh': round(float(row['median_delta']), 4),
        'pct_significant_fdr': round(float(row['pct_significant']), 1),
        'pct_increasing': round(float(row['pct_increasing']), 1),
        'n_class_B': int(row['n_class_B']),
        'n_class_C': int(row['n_class_C']),
        't_test_pvalue': round(float(row['t_pval']), 6) if not np.isnan(row['t_pval']) else None,
    }

summary_path = output_dir / 'dif_dimension_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\u2713 Dimension summary saved to: {summary_path}")

print("\n\u2713 Analysis complete!")